In [2]:

from pyspark.sql import SparkSession


In [3]:
spark =SparkSession.builder \
        .appName("odais app") \
        .master("local[*]") \
        .getOrCreate()

## Create Data

In [4]:

employees = spark.createDataFrame([
    (1, "Ali", 3000),
    (2, "Sara", 7000),
    (3, "Omar", None),
    (4, "Mona", 10000)
], ["id", "name", "salary"])

departments = spark.createDataFrame([
    (1, "HR"),
    (2, "IT"),
    (2, "IT_DUPLICATE"),
    (3, "Finance"),
], ["id", "department"])

employees.show()
departments.show()


+---+----+------+
| id|name|salary|
+---+----+------+
|  1| Ali|  3000|
|  2|Sara|  7000|
|  3|Omar|  null|
|  4|Mona| 10000|
+---+----+------+

+---+------------+
| id|  department|
+---+------------+
|  1|          HR|
|  2|          IT|
|  2|IT_DUPLICATE|
|  3|     Finance|
+---+------------+




## Join the two dataframe


In [28]:
# ربط الجدولين بناءً على عمود id
df_joined = employees.join(departments, on="id", how="inner")
df_joined.show()


+---+----+------+------------+
| id|name|salary|  department|
+---+----+------+------------+
|  1| Ali|  3000|          HR|
|  3|Omar|  null|     Finance|
|  2|Sara|  7000|          IT|
|  2|Sara|  7000|IT_DUPLICATE|
+---+----+------+------------+




### Debug Task:
- Why does id=2 appear multiple times?
###### It caused a One-to-Many match. id=2 has one row in the employees table but two rows in the departments table. Spark matches the single employee record with every available department record, duplicating the row in the final output.What kind of data issue caused this?Duplicate Data (Redundancy): The id column in the reference table (departments) is not unique.Data Inconsistency: The same ID is linked to two different values (IT and IT_DUPLICATE), which breaks the primary key integrity rule.If you want to fix this, let me know if you would like to drop duplicates or use a Window function to keep a specific row.
- What kind of data issue caused this?
###### Duplicate Data (Redundancy): The id column in the reference table (departments) is not unique.Data Inconsistency: The same ID is linked to two different values (IT and IT_DUPLICATE), which breaks the primary key integrity rule.

## Fix Join Issue

##### solution 1 
###### in my opinion : broadcasting , it's more efficient also becuase the mapping dic dont allow 2 same keys so will not be any duplicats

In [20]:
department_mapping= {
    1: "HR",
    2: "IT",
    2: "IT_DUPLICATE",
    3: "Finance"
}

In [21]:
sc = spark.sparkContext

In [22]:
broadcast_dept = sc.broadcast(department_mapping)

In [23]:
def get_dept_name(emp_id):
    return broadcast_dept.value.get(emp_id)

In [25]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

deprt_udf = udf(get_dept_name, StringType())

In [27]:
df_emp_2 = employees.withColumn("dept_name", deprt_udf(col('id')))
df_emp_2.show()


+---+----+------+------------+
| id|name|salary|   dept_name|
+---+----+------+------------+
|  1| Ali|  3000|          HR|
|  2|Sara|  7000|IT_DUPLICATE|
|  3|Omar|  null|     Finance|
|  4|Mona| 10000|        null|
+---+----+------+------------+



##### solution 2 
###### drop any duplicats from category table before joining


In [29]:
departments_clean = departments.dropDuplicates(["id"])

df_joined_correct = employees.join(departments_clean, on="id", how="inner")
df_joined_correct.show()


+---+----+------+----------+
| id|name|salary|department|
+---+----+------+----------+
|  1| Ali|  3000|        HR|
|  3|Omar|  null|   Finance|
|  2|Sara|  7000|        IT|
+---+----+------+----------+



## categorize the employee salary with new column using Case When

In [30]:
from pyspark.sql.functions import col, when

df_categorized = employees.withColumn(
    "salary_category",
    when(col("salary").isNull(), "Unknown")
    .when(col("salary") < 5000, "Low")
    .when(col("salary") <= 8000, "Medium")
    .otherwise("High")
)

df_categorized.show()



+---+----+------+---------------+
| id|name|salary|salary_category|
+---+----+------+---------------+
|  1| Ali|  3000|            Low|
|  2|Sara|  7000|         Medium|
|  3|Omar|  null|        Unknown|
|  4|Mona| 10000|           High|
+---+----+------+---------------+

